# 실습 1 — BERT 풀 파인튜닝 (감성 분석)

사전학습된 **DistilBERT** 모델을 IMDB 영화 리뷰 데이터로 파인튜닝하여 감성 분류 모델을 만듭니다.

| 항목 | 내용 |
|---|---|
| **모델** | `distilbert-base-uncased` (66M 파라미터) |
| **데이터셋** | IMDB 영화 리뷰 (50개 샘플) |
| **태스크** | 감성 분류 (긍정 / 부정) |
| **방법** | Full Fine-tuning (전체 파라미터 학습) |

> **인코더 모델 vs 디코더 모델**  
> DistilBERT는 인코더 전용 모델로, 텍스트를 생성하는 대신 입력 문장 전체를 한 번에 이해하는 데 특화되어 있습니다.  
> 분류, 감성 분석 같은 태스크에 적합합니다.

## ⚙️ Colab GPU 설정

실습 전 반드시 GPU를 활성화해야 합니다.

**설정 방법:**
1. 상단 메뉴 → **런타임** 클릭
2. **런타임 유형 변경** 클릭
3. 하드웨어 가속기 → **T4 GPU** 선택
4. **저장** 후 런타임 재시작

> ⚠️ GPU 없이 실행하면 학습 속도가 매우 느립니다. 반드시 설정 후 시작하세요.

In [ ]:
# GPU 활성화 확인 — 반드시 GPU가 표시되어야 합니다
import torch

print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 모델  : {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU가 활성화되지 않았습니다. 위의 설정 방법을 따라주세요.")

## 📦 라이브러리 설치

| 라이브러리 | 역할 |
|---|---|
| `transformers` | 사전학습 모델 로딩 및 파인튜닝 도구 |
| `datasets` | HuggingFace 데이터셋 로딩 |
| `scikit-learn` | 정확도(accuracy) 계산 |

In [ ]:
!pip install -q transformers datasets scikit-learn
print("✅ 설치 완료")

## 1단계. 데이터 준비

**IMDB 데이터셋**은 영화 리뷰와 감성 레이블(긍정/부정)로 구성된 NLP 대표 데이터셋입니다.

- 전체 크기: 학습 25,000개 / 테스트 25,000개
- 이번 실습: 빠른 실습을 위해 **50개만** 사용 (학습 40개 / 테스트 10개)

> **왜 데이터를 나눌까?**  
> 학습 데이터로 모델을 훈련하고, 테스트 데이터로 모델이 새로운 입력을 얼마나 잘 처리하는지 검증합니다.  
> 학습에 사용한 데이터로 테스트하면 모델이 답을 외운 것인지 실제로 이해한 것인지 구별할 수 없습니다.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb", split="train")

# 먼저 전체 train을 섞고 나서 500개 선택
dataset = dataset.shuffle(seed=42).select(range(500))

# 그 다음 train/test split
dataset = dataset.train_test_split(test_size=0.2, seed=42)

from collections import Counter
print("train label 분포:", Counter(dataset["train"]["label"]))
print("test label 분포 :", Counter(dataset["test"]["label"]))

print(f"학습 데이터 수: {len(dataset['train'])}")
print(f"테스트 데이터 수: {len(dataset['test'])}")

In [ ]:
# 샘플 하나 확인
sample = dataset["train"][0]
print("[리뷰 내용 앞 300자]")
print(sample['text'][:300])
print(f"\n레이블: {sample['label']} → {'긍정(Positive)' if sample['label'] == 1 else '부정(Negative)'}")

## 2단계. 토크나이저 및 모델 준비

**토크나이저(Tokenizer)**는 텍스트를 모델이 이해할 수 있는 숫자(토큰 ID)로 변환합니다.

```
"I love this movie" → [101, 1045, 2293, 2023, 3185, 102]
                           ↑                              ↑
                       [CLS] 시작 토큰            [SEP] 끝 토큰
```

> 반드시 **모델과 같은 토크나이저**를 사용해야 합니다.  
> 토크나이저가 다르면 같은 단어도 다른 숫자로 변환되어 모델이 제대로 동작하지 않습니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# num_labels=2: 분류할 클래스 수 (부정=0, 긍정=1)

print(f"✅ 모델 로딩 완료: {MODEL_NAME}")
print(f"전체 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

## 3단계. 데이터 전처리 (텍스트 → 토큰)

---
### ✏️ 빈칸 1, 2 — tokenize 함수 완성

`tokenizer()` 함수의 두 옵션을 채워보세요.

| 옵션 | 역할 | 채울 값 |
|---|---|---|
| `truncation` | 512 토큰을 초과하는 문장을 잘라냄 | `True` 또는 `False` |
| `padding` | 짧은 문장을 0으로 채워 길이를 통일 | `True` 또는 `False` |

**힌트:** 두 옵션 모두 활성화해야 배치(여러 문장을 한 번에) 처리가 가능합니다.

---

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,  # ✏️ 빈칸 1
        padding=True,     # ✏️ 빈칸 2
    )

dataset = dataset.map(tokenize, batched=True)

print("✅ 전처리 완료")
print(f"input_ids 예시: {dataset['train'][0]['input_ids'][:10]} ...")
print(f"토큰 총 개수  : {len(dataset['train'][0]['input_ids'])}개")

### 📊 전처리 결과 해석

- **`input_ids`**: 각 단어/서브워드에 부여된 번호
- **`attention_mask`**: 실제 토큰(1)과 패딩 자리(0)를 구분
- `101`은 문장 시작([CLS]), `102`는 문장 끝([SEP]), `0`은 패딩을 의미합니다.

## 4단계. 훈련 설정

---
### ✏️ 빈칸 3, 4, 5 — 하이퍼파라미터와 metrics 함수 완성

아래 표를 참고해서 빈칸을 채워보세요.

| 파라미터 | 권장값 | 설명 |
|---|---|---|
| `per_device_train_batch_size` | `8` | GPU 하나가 한 번에 처리할 샘플 수 |
| `num_train_epochs` | `15` | 전체 데이터를 반복 학습하는 횟수 |
| `argmax` 메서드 | `argmax` | logits에서 가장 높은 점수의 인덱스 추출 |

---

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # ✏️ 빈칸 3: logits에서 가장 높은 점수의 인덱스를 예측값으로 사용
    # logits shape: (batch_size, 2) → [부정 점수, 긍정 점수]
    predictions = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

args = TrainingArguments(
    output_dir="./bert-finetuned",
    per_device_train_batch_size=8,  # ✏️ 빈칸 4: 배치 크기
    num_train_epochs=5,             # ✏️ 빈칸 5: 에폭 수
    report_to="none",
    logging_steps=1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

print("✅ Trainer 구성 완료")

## 5단계. 파인튜닝 실행

`trainer.train()` 한 줄로 아래 과정이 에폭마다 자동 반복됩니다.

```
입력 데이터 → 순전파(Forward) → 손실(Loss) 계산 → 역전파(Backward) → 파라미터 업데이트
```

학습 중 **`loss` 값이 점점 줄어드는** 것을 확인해보세요.

In [ ]:
trainer.train()

### 📊 학습 결과 해석

- **`loss`** 값이 낮아질수록 모델이 훈련 데이터를 잘 학습하고 있다는 의미입니다.
- 50개의 작은 데이터셋이라 loss가 0에 가까워질 수 있는데, 이는 **과적합(Overfitting)** 신호일 수 있습니다.
  - 과적합: 훈련 데이터는 잘 맞히지만 새로운 데이터는 못 맞히는 현상
- 실제 서비스에서는 수천~수만 개의 데이터로 학습합니다.

## 6단계. 새 문장 예측

---
### ✏️ 빈칸 6, 7, 8 — 예측 코드 완성

---

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ✏️ 빈칸 6: 모델을 평가(추론) 모드로 전환
# 힌트: 학습 모드(train)와 반대 모드의 이름은?
# → 평가 모드에서는 Dropout 같은 랜덤 요소가 꺼져 예측이 일관됩니다
model.eval()

text = "I would put this at the top of my list of films in the category of unwatchable trash!"
inputs = tokenizer(text, return_tensors="pt").to(device)

# torch.no_grad(): 추론 시 gradient 계산을 끄면 메모리가 절약됩니다
with torch.no_grad():
    output = model(**inputs)
    # ✏️ 빈칸 7: logits에서 예측 클래스 추출 (빈칸 3과 같은 메서드)
    label = output.logits.argmax(-1).item()

# ✏️ 빈칸 8: 긍정 레이블의 번호 (0=부정, 1=긍정)
print(f"예측 결과: {'긍정 ✅' if label == 1 else '부정 ❌'}")
print(f"리뷰: {text[:60]}...")

In [ ]:
# 여러 문장으로 직접 테스트해보기
test_texts = [
    "This movie was absolutely wonderful! A masterpiece.",
    "What a waste of time. Terrible acting and a boring plot.",
    "It was okay, nothing special but not bad either.",
]

print("=" * 60)
for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model(**inputs)
        label = output.logits.argmax(-1).item()
    result = "긍정 ✅" if label == 1 else "부정 ❌"
    print(f"{result} | {text[:55]}...")
print("=" * 60)

## ✅ 실습 1 완료!

### 전체 흐름 정리

```
데이터 로딩 → 토크나이징 → 모델 로딩 → Trainer 구성 → 학습 → 예측
```

| 단계 | 핵심 코드 | 역할 |
|---|---|---|
| 전처리 | `tokenizer(text, truncation=True, padding=True)` | 텍스트 → 토큰 변환 |
| 학습 | `trainer.train()` | 파인튜닝 실행 |
| 예측 | `logits.argmax(-1)` | 클래스 예측 |

### 다음 실습
실습 2에서는 더 큰 생성형 모델(Qwen2.5-3B)을 **LoRA**로 효율적으로 파인튜닝합니다.  
전체 파라미터의 단 **1%만 학습**해도 좋은 성능을 낼 수 있습니다!